### Bibliotecas

In [1]:
import pandas as pd

import warnings
import os

import matplotlib.pyplot as plt
import seaborn as sns

### Configurações

In [ ]:
# Definindo diretporio
os.chdir("")

# Exibir todas as linhas e colunas
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# evitar a exibição de avisos
warnings.filterwarnings("ignore")

In [3]:
# Importando os dados

# Treino
train = pd.read_csv("train.csv")

# Teste
test = pd.read_csv("test.csv")

# Submissão
sample_submission = pd.read_csv("sample_submission.csv")

### EDA

In [4]:
# Primeiras observações
print(test.head())

      id  Time_spent_Alone Stage_fear  Social_event_attendance  Going_outside  \
0  18524               3.0         No                      7.0            4.0   
1  18525               NaN        Yes                      0.0            0.0   
2  18526               3.0         No                      5.0            6.0   
3  18527               3.0         No                      4.0            4.0   
4  18528               9.0        Yes                      1.0            2.0   

  Drained_after_socializing  Friends_circle_size  Post_frequency  
0                        No                  6.0             NaN  
1                       Yes                  5.0             1.0  
2                        No                 15.0             9.0  
3                        No                  5.0             6.0  
4                       Yes                  1.0             1.0  


In [5]:
# Dimenções do df
print(f"Dimensões do DataFrame de treino: {train.shape[0]} linhas, {train.shape[1]} colunas")
print(f"Dimensões do DataFrame de teste : {test.shape[0]} linhas, {test.shape[1]} colunas")

Dimensões do DataFrame de treino: 18524 linhas, 9 colunas
Dimensões do DataFrame de teste : 6175 linhas, 8 colunas


In [6]:
# Resumo do DF
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18524 entries, 0 to 18523
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         18524 non-null  int64  
 1   Time_spent_Alone           17334 non-null  float64
 2   Stage_fear                 16631 non-null  object 
 3   Social_event_attendance    17344 non-null  float64
 4   Going_outside              17058 non-null  float64
 5   Drained_after_socializing  17375 non-null  object 
 6   Friends_circle_size        17470 non-null  float64
 7   Post_frequency             17260 non-null  float64
 8   Personality                18524 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 1.3+ MB


In [7]:
# Quantidade dados faltantes
train.isna().mean()*100

id                            0.000000
Time_spent_Alone              6.424098
Stage_fear                   10.219175
Social_event_attendance       6.370114
Going_outside                 7.914057
Drained_after_socializing     6.202764
Friends_circle_size           5.689916
Post_frequency                6.823580
Personality                   0.000000
dtype: float64

In [8]:
train.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


### Separar em treino e validação

In [9]:
from sklearn.model_selection import train_test_split

# Separando em treino e teste
target = "Personality"
X = train.drop(columns=[target, 'id'])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True)

In [10]:
print(y_train.head())

11123    Extrovert
360      Introvert
15844    Extrovert
15624    Extrovert
6466     Extrovert
Name: Personality, dtype: object


### Substituindo dados faltantes

In [11]:
from sklearn.impute import SimpleImputer

# Criar o imputador para substituir pela moda (most frequent value)
imputer = SimpleImputer(strategy='most_frequent')

# Ajustar (fit) nos dados de treino e transformar (impute) X_train
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Usar o mesmo imputador treinado para transformar X_valid
X_valid_imputed = pd.DataFrame(imputer.transform(X_valid), columns=X_valid.columns)

In [12]:
X_train_imputed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13893 entries, 0 to 13892
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Time_spent_Alone           13893 non-null  object
 1   Stage_fear                 13893 non-null  object
 2   Social_event_attendance    13893 non-null  object
 3   Going_outside              13893 non-null  object
 4   Drained_after_socializing  13893 non-null  object
 5   Friends_circle_size        13893 non-null  object
 6   Post_frequency             13893 non-null  object
dtypes: object(7)
memory usage: 759.9+ KB


### Criando dummies

In [13]:
# Transformar em dummies com drop_first=True no treino
X_train_dummies = pd.get_dummies(X_train_imputed, drop_first=True)

# Garantir que valid tenha as mesmas colunas do treino
X_valid_dummies = pd.get_dummies(X_valid_imputed, drop_first=True)

# Alinhar as colunas (reindexar o conjunto de validação com as colunas do treino)
X_valid_dummies = X_valid_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

# Alinhar X_train_dummies com X_valid_dummies caso queira simetria total:
X_train_dummies = X_train_dummies.reindex(columns=X_valid_dummies.columns, fill_value=0)

In [14]:
X_train_dummies = X_train_dummies.astype(float)
X_valid_dummies = X_valid_dummies.astype(float)

### Codificando

In [15]:
from sklearn.preprocessing import LabelEncoder

# Codificar os rótulos (Introvert = 0, Extrovert = 1)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_valid_encoded = le.transform(y_valid)

### Previsão

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

##### DecisionTree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Criar pipeline: SMOTE + Decision Tree
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Grade de hiperparâmetros (referência ao passo "classifier")
param_grid = {
    'classifier__criterion': ['gini', 'entropy', 'log_loss'],  # critérios de impureza
    'classifier__max_depth': [3, 5, 10, 15, None],             # profundidade máxima
    'classifier__min_samples_split': [2, 5, 10, 20],           # mínimo para dividir um nó
    'classifier__min_samples_leaf': [1, 2, 4, 8],              # mínimo por folha
    'classifier__max_features': [None, 'sqrt', 'log2']         # estratégia para selecionar atributos
}

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=3
)

# Treinar (SMOTE será aplicado dentro do CV, somente no treino)
grid_search.fit(X_train_dummies, y_train_encoded)

# Melhor modelo
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliar no conjunto de validação
y_pred = best_model.predict(X_valid_dummies)
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

'from sklearn.tree import DecisionTreeClassifier\nfrom sklearn.model_selection import GridSearchCV\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom imblearn.over_sampling import SMOTE\nfrom imblearn.pipeline import Pipeline\n\n# Criar pipeline: SMOTE + Decision Tree\npipeline = Pipeline([\n    (\'smote\', SMOTE(random_state=42)),\n    (\'classifier\', DecisionTreeClassifier(random_state=42))\n])\n\n# Grade de hiperparâmetros (referência ao passo "classifier")\nparam_grid = {\n    \'classifier__criterion\': [\'gini\', \'entropy\', \'log_loss\'],  # critérios de impureza\n    \'classifier__max_depth\': [3, 5, 10, 15, None],             # profundidade máxima\n    \'classifier__min_samples_split\': [2, 5, 10, 20],           # mínimo para dividir um nó\n    \'classifier__min_samples_leaf\': [1, 2, 4, 8],              # mínimo por folha\n    \'classifier__max_features\': [None, \'sqrt\', \'log2\']         # estratégia para selecionar atributos\n}\n\n# Configurar GridS

##### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Criar pipeline: SMOTE + RandomForest
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Grade de hiperparâmetros (enxuta, mas relevante)
param_grid = {
    'classifier__n_estimators': [100, 200],        # nº de árvores
    'classifier__max_depth': [5, 10, None],        # profundidade máxima
    'classifier__min_samples_split': [2, 5],       # mínimo para dividir nó
    'classifier__min_samples_leaf': [1, 2],        # mínimo por folha
    'classifier__max_features': ['sqrt', 'log2']   # nº máximo de atributos por split
}

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=3
)

# Treinar (SMOTE será aplicado dentro do CV, somente no treino)
grid_search.fit(X_train_dummies, y_train_encoded)

# Melhor modelo
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliar no conjunto de validação
y_pred = best_model.predict(X_valid_dummies)
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

'from sklearn.ensemble import RandomForestClassifier\nfrom sklearn.model_selection import GridSearchCV\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom imblearn.over_sampling import SMOTE\nfrom imblearn.pipeline import Pipeline\n\n# Criar pipeline: SMOTE + RandomForest\npipeline = Pipeline([\n    (\'smote\', SMOTE(random_state=42)),\n    (\'classifier\', RandomForestClassifier(random_state=42))\n])\n\n# Grade de hiperparâmetros (enxuta, mas relevante)\nparam_grid = {\n    \'classifier__n_estimators\': [100, 200],        # nº de árvores\n    \'classifier__max_depth\': [5, 10, None],        # profundidade máxima\n    \'classifier__min_samples_split\': [2, 5],       # mínimo para dividir nó\n    \'classifier__min_samples_leaf\': [1, 2],        # mínimo por folha\n    \'classifier__max_features\': [\'sqrt\', \'log2\']   # nº máximo de atributos por split\n}\n\n# Configurar GridSearchCV\ngrid_search = GridSearchCV(\n    estimator=pipeline,\n    param_grid=param_grid,

##### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Pipeline: Normalização + SMOTE + KNN
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # KNN sensível à escala
    ('smote', SMOTE(random_state=42)), # Balanceamento
    ('classifier', KNeighborsClassifier())
])

# Grade de hiperparâmetros (moderada)
param_grid = {
    'classifier__n_neighbors': [3, 5, 7],          # nº de vizinhos
    'classifier__weights': ['uniform', 'distance'],# pesos
    'classifier__metric': ['euclidean', 'manhattan'] # métricas de distância
}

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=3
)

# Treinar
grid_search.fit(X_train_dummies, y_train_encoded)

# Melhor modelo
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliar no conjunto de validação
y_pred = best_model.predict(X_valid_dummies)
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

'from sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.model_selection import GridSearchCV\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom imblearn.over_sampling import SMOTE\nfrom imblearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\n# Pipeline: Normalização + SMOTE + KNN\npipeline = Pipeline([\n    (\'scaler\', StandardScaler()),      # KNN sensível à escala\n    (\'smote\', SMOTE(random_state=42)), # Balanceamento\n    (\'classifier\', KNeighborsClassifier())\n])\n\n# Grade de hiperparâmetros (moderada)\nparam_grid = {\n    \'classifier__n_neighbors\': [3, 5, 7],          # nº de vizinhos\n    \'classifier__weights\': [\'uniform\', \'distance\'],# pesos\n    \'classifier__metric\': [\'euclidean\', \'manhattan\'] # métricas de distância\n}\n\n# Configurar GridSearchCV\ngrid_search = GridSearchCV(\n    estimator=pipeline,\n    param_grid=param_grid,\n    scoring=\'accuracy\',\n    cv=5,\n    verbose=3\n)\n\n# Treinar

##### XGB

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Pipeline: SMOTE + XGBoost
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)), # Balanceamento
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# Grade de hiperparâmetros (moderada)
param_grid = {
    'classifier__n_estimators': [100, 200],       # número de árvores
    'classifier__max_depth': [3, 5, 7],           # profundidade das árvores
    'classifier__learning_rate': [0.01, 0.1],     # taxa de aprendizado
    'classifier__subsample': [0.8, 1.0]           # amostragem para cada árvore
}

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=3
)

# Treinar
grid_search.fit(X_train_dummies, y_train_encoded)

# Melhor modelo
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliar no conjunto de validação
y_pred = best_model.predict(X_valid_dummies)
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

'from xgboost import XGBClassifier\nfrom sklearn.model_selection import GridSearchCV\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom imblearn.over_sampling import SMOTE\nfrom imblearn.pipeline import Pipeline\n\n# Pipeline: SMOTE + XGBoost\npipeline = Pipeline([\n    (\'smote\', SMOTE(random_state=42)), # Balanceamento\n    (\'classifier\', XGBClassifier(use_label_encoder=False, eval_metric=\'logloss\', random_state=42))\n])\n\n# Grade de hiperparâmetros (moderada)\nparam_grid = {\n    \'classifier__n_estimators\': [100, 200],       # número de árvores\n    \'classifier__max_depth\': [3, 5, 7],           # profundidade das árvores\n    \'classifier__learning_rate\': [0.01, 0.1],     # taxa de aprendizado\n    \'classifier__subsample\': [0.8, 1.0]           # amostragem para cada árvore\n}\n\n# Configurar GridSearchCV\ngrid_search = GridSearchCV(\n    estimator=pipeline,\n    param_grid=param_grid,\n    scoring=\'accuracy\',\n    cv=5,\n    verbose=3\n)\n\n# Tre

### Stack - Modelo Final

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Converter para float
X_train_dummies = X_train_dummies.astype(float)
X_valid_dummies = X_valid_dummies.astype(float)

# Modelos base com melhores parâmetros
decision_tree_best = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=3,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

random_forest_best = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    max_features='log2',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

knn_best = KNeighborsClassifier(
    n_neighbors=7,
    weights='uniform',
    metric='manhattan'
)

xgb_best = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Stacking com meta-modelo Logistic Regression
stacking_clf = StackingClassifier(
    estimators=[
        ('decision_tree', decision_tree_best),
        ('random_forest', random_forest_best),
        ('knn', knn_best),
        ('xgb', xgb_best)
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5
)

# Pipeline com SMOTE + Scaling (para KNN)
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Normalização para KNN
    ('smote', SMOTE(random_state=42)), # Balanceamento
    ('stacking', stacking_clf)
])

# Treinar o modelo
pipeline.fit(X_train_dummies, y_train_encoded)

# Avaliar no conjunto de validação
y_pred = pipeline.predict(X_valid_dummies)

# Métricas
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

'from sklearn.ensemble import StackingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom xgboost import XGBClassifier\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom imblearn.pipeline import Pipeline\nfrom imblearn.over_sampling import SMOTE\nfrom sklearn.preprocessing import StandardScaler\n\n# Converter para float\nX_train_dummies = X_train_dummies.astype(float)\nX_valid_dummies = X_valid_dummies.astype(float)\n\n# Modelos base com melhores parâmetros\ndecision_tree_best = DecisionTreeClassifier(\n    criterion=\'entropy\',\n    max_depth=3,\n    max_features=\'sqrt\',\n    min_samples_leaf=1,\n    min_samples_split=2,\n    random_state=42\n)\n\nrandom_forest_best = RandomForestClassifier(\n    n_estimators=200,\n    max_depth=5,\n    max_features=\'log2\',\n    min_samples_leaf=1,\n    mi

##### ExtraTree

In [34]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Pipeline: SMOTE + ExtraTrees
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),  # Balanceamento
    ('classifier', ExtraTreesClassifier(random_state=42))
])

# Grade de hiperparâmetros (apenas os 3 principais)
param_grid = {
    'classifier__n_estimators': [100, 200],  # número de árvores
    'classifier__max_depth': [None, 10, 20], # profundidade máxima
    'classifier__max_features': ['sqrt', 'log2'] # número de features
}

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=3
)

# Treinar
grid_search.fit(X_train_dummies, y_train_encoded)

# Melhor modelo
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliar no conjunto de validação
y_pred = best_model.predict(X_valid_dummies)
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100;, score=0.960 total time=   3.3s
[CV 2/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100;, score=0.960 total time=   3.0s
[CV 3/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100;, score=0.959 total time=   2.7s
[CV 4/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100;, score=0.956 total time=   2.7s
[CV 5/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100;, score=0.965 total time=   3.1s
[CV 1/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=200;, score=0.960 total time=   5.6s
[CV 2/5] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=200;, score=0.960 total time=   6.0s
[

### HistGradient

In [40]:
from sklearn.experimental import enable_hist_gradient_boosting  # necessário em algumas versões
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Pipeline: SMOTE + HistGradientBoosting
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),  # Balanceamento
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

# Grade de hiperparâmetros (3 principais)
param_grid = {
    'classifier__max_iter': [100, 200],    # número de iterações (similar a n_estimators)
    'classifier__learning_rate': [0.01, 0.1], # taxa de aprendizado
    'classifier__max_depth': [3, 5, 7]     # profundidade máxima
}

# Configuração do GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=3
)

# Treinar
grid_search.fit(X_train_dummies, y_train_encoded)

# Melhor modelo
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliar no conjunto de validação
y_pred = best_model.predict(X_valid_dummies)
print("Acurácia no conjunto de validação:", accuracy_score(y_valid_encoded, y_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=100;, score=0.965 total time=   0.6s
[CV 2/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=100;, score=0.969 total time=   0.5s
[CV 3/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=100;, score=0.966 total time=   0.5s
[CV 4/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=100;, score=0.967 total time=   0.5s
[CV 5/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=100;, score=0.974 total time=   0.6s
[CV 1/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=200;, score=0.965 total time=   0.9s
[CV 2/5] END classifier__learning_rate=0.01, classifier__max_depth=3, classifier__max_iter=200;, score=0.969 total time=   1.2s
[CV 3/5] END classifier__learning_rate=0.01

### Teste

##### Dados faltantes

In [35]:
# Imputar dados faltantes no teste
X_test_imputed = pd.DataFrame(imputer.transform(test.drop(columns=['id'])), columns=X.columns)

##### Dummies

In [36]:
# Transformar em dummies e alinhar com treino
X_test_dummies = pd.get_dummies(X_test_imputed, drop_first=True)
X_test_dummies = X_test_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)
X_test_dummies = X_test_dummies.astype(int)

##### Prevendo

In [37]:
# Prever no conjunto de teste (em formato codificado)
test_preds_encoded = best_model.predict(X_test_dummies)

##### Codificação

In [38]:
# Reverter para rótulos originais
test_preds = le.inverse_transform(test_preds_encoded)

### Submissão

In [39]:
sample_submission['Personality'] = test_preds
sample_submission.to_csv("extratree.csv", index=False)